### When to use **ENACTING** vs **INTERACTING**


#### ENACTING 
- **What it is:** Students **enact roles** (e.g. molecules) in a **physical + projected** photosynthesis task: **movement, action, state, and gaze (to the screen)** carry meaning. You label segments of **ENACTING** (the target construct), not general classroom talk.
- **Prompt file:** `Prompt_ENACTING.txt` → build few-shot with `enacting_fewshot_messages(model_id)` → final chat payload is **`messages`** (few-shot + MESSI-style CSV as the last user turn).
- **CSV shaping:** In code, pass **`action="enacting"`** to `prepare_embodied_csv_for_action(df, "enacting", student_name)`. That **drops** rows with modality **`gesture`** and **`speech`**; for **`gaze`**, **keeps only** rows where **`data == "Screen"`** (other gaze targets are removed). Replaces `not moving` → `stationary` when a `data` column exists.
- **Notebook path:** Section **3. ENACTING branch** — mastersheet load → `data_csv_string` / **`messages`** → optional four-model cell with **`RUN_ENACTING_FOUR_MODELS = True`**.

#### INTERACTING 
- **What it is:** **Teacher–student dialogue** and related modalities: **speech**, **state**, **gaze** matter; **full-body movement / action / gesture** streams are **not** part of this labeling format (they are stripped before the API call).
- **Prompt file:** `Prompt_INTERACTING.txt` → `interacting_fewshot_messages(model_id)` → final payload is **`messages_interacting`** (few-shot + formatted CSV as the last user turn).
- **CSV shaping:** Pass **`action="interacting"`** to `prepare_embodied_csv_for_action(df, "interacting", student_name)`. That **drops** **`movement`**, **`action`**, and **`gesture`**. The text sent to the model is prefixed with **`Speaker: <student_name>`** and a blank line before the CSV body (Colab / `L_and_I_Interacting_Script` style).
- **Notebook path:** Section **4. INTERACTING branch** — few-shot cell, then **INTERACTING · Rose day 1** builds **`messages_interacting`** → optional four-model cell with **`RUN_INTERACTING_FOUR_MODELS = True`**.

#### Batch jobs (many files)
For each task dict `t`, call **`prepare_embodied_csv_for_action(df, t["action"], t["student"])`** with **`t["action"]` in `{"enacting", "interacting"}`** so CSV formatting matches the behavior type. Then attach the returned body to the correct message list (**`messages`** vs **`messages_interacting`**) and the matching prompt file. *For many files, copy the same pattern as sections 3–4 in your own loop — no separate pipeline file is required.*


## 1. Shared — paths & CSV helper

`PROJECT_ROOT`, `SPLIT_STRING`, `STUDENT_DAY_CSV_DIR`, and `prepare_embodied_csv_for_action` — run this before any ENACTING or INTERACTING few-shot cells.


In [18]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT")

# Shared delimiter in Prompt_ENACTING.txt and Prompt_INTERACTING.txt
SPLIT_STRING = "\n[***NEW_MESSAGE***]\n"

STUDENT_DAY_CSV_DIR = PROJECT_ROOT / "study-data-per-student-day-behavior"


def prepare_embodied_csv_for_action(df, action: str, student_name: str):
    """Normalize a modality CSV for the API.

    action == "enacting": drop gesture & speech; keep gaze rows only when data == "Screen".
    action == "interacting": drop movement, action, gesture; prefix CSV text with Speaker: <student_name>.
    """
    import pandas as pd

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    d = df.copy().fillna("")
    if "data" in d.columns:
        d["data"] = d["data"].replace("not moving", "stationary")

    if action == "enacting":
        d = d[~d["modality"].isin(["gesture", "speech"])]
        d = d[(d["modality"] != "gaze") | (d["data"] == "Screen")]
    elif action == "interacting":
        d = d[~d["modality"].isin(["movement", "action", "gesture"])]
    else:
        d = d

    body = d.to_csv(index=False)
    if action == "interacting":
        body = f"Speaker: {student_name}\n\n" + body
    return body, d


## 2. Shared — API token & headers

Run **Configuration** before `get_available_models` / `run_pipeline`. `PREVIEW_MODEL_ID` is set there for the single-file `messages` / `messages_interacting` cells.


In [19]:
## Configuration
import os
import requests

BASE_URL = "https://prod-api.vanderbilt.ai"
# Paste token here, or leave empty and use shell env: export AMP_TOKEN='…'

if AMP_TOKEN.strip():
    os.environ["AMP_TOKEN"] = AMP_TOKEN.strip()
AMP_TOKEN = os.environ.get("AMP_TOKEN", "").strip()
if not AMP_TOKEN:
    raise RuntimeError(
        "Set AMP_TOKEN: paste into AMP_TOKEN = '' in this cell, or export AMP_TOKEN='…', then re-run."
    )

headers = {
    "Authorization": f"Bearer {AMP_TOKEN}",
    "Content-Type": "application/json",
}

# Model id used when composing `messages` / `messages_interacting` in the single-file cells below.
PREVIEW_MODEL_ID = "gpt-5.2"

_ar = requests.get(f"{BASE_URL.rstrip('/')}/available_models", headers=headers, timeout=30)
print("Config ready for pipeline. available_models:", _ar.status_code, end=" ")
if _ar.status_code == 200:
    print("model_count:", len(_ar.json().get("data", {}).get("models", [])))
else:
    print(_ar.text[:300])


Config ready for pipeline. available_models: 200 model_count: 12


## 3. ENACTING branch

Few-shot → mastersheet import → default **`messages`** for ENACTING runs.


In [20]:
# ENACTING: few-shot prefix from Prompt_ENACTING.txt (use with `messages` + MESSI CSV below).
PROMPT_PATH_ENACTING = PROJECT_ROOT / "Prompt_ENACTING.txt"
with open(PROMPT_PATH_ENACTING, "r", encoding="utf-8") as file:
    prompt_raw_enacting = file.read()

prompt_split_enacting = prompt_raw_enacting.split(SPLIT_STRING)
assert len(prompt_split_enacting) == 3, "Prompt file format incorrect (system + demo user + demo assistant)."


def enacting_fewshot_messages(model_id: str) -> list:
    """OpenAI: system + user + assistant. Claude: user + user + assistant."""
    mid = (model_id or "").lower()
    claude = "claude" in mid or "anthropic" in mid
    if claude:
        return [
            {"role": "user", "content": prompt_split_enacting[0].strip()},
            {"role": "user", "content": prompt_split_enacting[1].strip()},
            {"role": "assistant", "content": prompt_split_enacting[2].strip()},
        ]
    return [
        {"role": "system", "content": prompt_split_enacting[0].strip()},
        {"role": "user", "content": prompt_split_enacting[1].strip()},
        {"role": "assistant", "content": prompt_split_enacting[2].strip()},
    ]



In [21]:
import pandas as pd

# ENACTING-only source: full-body / screen task modalities (not the interacting discourse CSV layout).
DATA_PATH = PROJECT_ROOT / "L&I - Student Mastersheet - EN-MESSI-D2.csv"

df = pd.read_csv(DATA_PATH)
df = df.fillna("")

df["data"] = df["data"].replace("not moving", "stationary")

df_no_speech_or_gaze = df[~df["modality"].isin(["gesture", "speech"])]
assert set(df_no_speech_or_gaze.modality) == {"state", "movement", "action", "gaze"}

df_no_speech_or_gaze_screen_only = df_no_speech_or_gaze[
    (df_no_speech_or_gaze["modality"] != "gaze") | (df_no_speech_or_gaze["data"] == "Screen")
]
assert all(
    df_no_speech_or_gaze_screen_only[df_no_speech_or_gaze_screen_only["modality"] == "gaze"]["data"] == "Screen"
)

data_csv_string = df_no_speech_or_gaze_screen_only.to_csv(index=False)

DEFAULT_BEHAVIOR_LABEL = "ENACTING"
OUTPUT_STEM_PREFIX = "MESSI_ENACT"
AMALIA_PROCESSOR_LABEL = "ENACTING MESSI"

df_no_speech_or_gaze_screen_only.head()


,start_time,end_time,modality,data
0,0:00:00,,state,waterthinking
1,0:00:00,0:00:05,movement,stationary
3,0:00:06,0:00:09,movement,moving
4,0:00:07,0:04:38,gaze,Screen
5,0:00:08,0:00:14,movement,stationary


**Note:** For **ENACTING**, row filtering and `data_csv_string` are already produced in **Import data (ENACTING mastersheet)**. You can skip re-deriving them here.


In [22]:
# ENACTING: few-shot (Prompt_ENACTING) + mastersheet CSV — this is the `messages` list for the default four-model cell.
messages = enacting_fewshot_messages(PREVIEW_MODEL_ID) + [
    {"role": "user", "content": data_csv_string},
]

print("Message count:", len(messages))

Message count: 4


## 4. INTERACTING branch

Few-shot → per-student discourse CSV → **`messages_interacting`** (Rose day 1 example).


### INTERACTING — few-shot from `Prompt_INTERACTING.txt`

Defines `interacting_fewshot_messages(model_id)` for **`messages_interacting`** (append the shaped discourse CSV as the final user turn in **INTERACTING · Rose day 1** below). Uses the same `[***NEW_MESSAGE***]` delimiter as the ENACTING cell above (`SPLIT_STRING`).

In [23]:
# INTERACTING: few-shot prefix from Prompt_INTERACTING.txt (use with `messages_interacting` + discourse CSV below).
PROMPT_PATH_INTERACTING = PROJECT_ROOT / "Prompt_INTERACTING.txt"
with open(PROMPT_PATH_INTERACTING, "r", encoding="utf-8") as file:
    prompt_raw_interacting = file.read()

prompt_split_interacting = prompt_raw_interacting.split(SPLIT_STRING)
assert len(prompt_split_interacting) == 3, "Prompt file format incorrect (system + demo user + demo assistant)."


def interacting_fewshot_messages(model_id: str) -> list:
    """OpenAI: system + user + assistant. Claude: user + user + assistant."""
    mid = (model_id or "").lower()
    claude = "claude" in mid or "anthropic" in mid
    if claude:
        return [
            {"role": "user", "content": prompt_split_interacting[0].strip()},
            {"role": "user", "content": prompt_split_interacting[1].strip()},
            {"role": "assistant", "content": prompt_split_interacting[2].strip()},
        ]
    return [
        {"role": "system", "content": prompt_split_interacting[0].strip()},
        {"role": "user", "content": prompt_split_interacting[1].strip()},
        {"role": "assistant", "content": prompt_split_interacting[2].strip()},
    ]


In [24]:
# INTERACTING example: load a per-student discourse CSV, shape with action="interacting", build `messages_interacting`.
# Few-shot prefix is defined above (**INTERACTING — few-shot from Prompt_INTERACTING.txt**); run that cell first.
import re
import pandas as pd
from pathlib import Path

ROSE_D1_INTERACTING_CSV = str(
    PROJECT_ROOT
    / "study-data-per-student-day-behavior"
    / "L&I - embodied - Student Rose - day 1 - interacting.csv"
)

_df_r = pd.read_csv(ROSE_D1_INTERACTING_CSV).fillna("")
_stem = Path(ROSE_D1_INTERACTING_CSV).stem
_m = re.search(r"Student\s+(.+?)\s+-\s+day", _stem)
_rose_name = _m.group(1).strip() if _m else "Rose"

api_csv_interacting, _df_inter_model = prepare_embodied_csv_for_action(_df_r, "interacting", _rose_name)

messages_interacting = interacting_fewshot_messages(PREVIEW_MODEL_ID) + [
    {"role": "user", "content": api_csv_interacting},
]

DEFAULT_BEHAVIOR_LABEL_INTER = "INTERACTING"
OUTPUT_STEM_PREFIX_INTER = "Rose_D1_INTER"
AMALIA_PROCESSOR_LABEL_INTER = "INTERACTING Rose D1"

print("INTERACTING message count:", len(messages_interacting))
_df_inter_model.head()


INTERACTING message count: 4


,start_time,end_time,modality,data
0,0:00:01,,speech,Teacher - Ms Hughes: Okay.
1,0:00:02,,speech,Teacher - Ms Hughes: So I want you to take a l...
2,0:00:05,0:00:10,gaze,Screen
3,0:00:05,,speech,Teacher - Ms Hughes: So what is Rose starting ...
4,0:00:08,,speech,Student - Rose: Air.


## 5. Inference pipeline

Run **in this order:** **Unified Inference Pipeline** (defines `run_pipeline`) → **Hotfix** (wraps it for JSON retries) → the **print** cell to confirm definitions.


In [25]:
# ===== Unified Inference Pipeline =====
import os
import re
import json
import time
import requests
import pandas as pd


def get_available_models(base_url: str, headers: dict):
    url = f"{base_url}/available_models"
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    payload = r.json()

    data = payload.get("data", {}) if isinstance(payload, dict) else {}
    models = data.get("models", []) if isinstance(data, dict) else []
    default_model = data.get("default", {}) if isinstance(data, dict) else {}

    model_ids = [m.get("id") for m in models if isinstance(m, dict) and isinstance(m.get("id"), str)]
    model_ids = list(dict.fromkeys(model_ids))
    default_id = default_model.get("id") if isinstance(default_model, dict) else None

    model_lookup = {
        m.get("id"): m for m in models
        if isinstance(m, dict) and isinstance(m.get("id"), str)
    }
    return model_ids, default_id, payload, model_lookup


def _unwrap_code_fence(text: str) -> str:
    s = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", s, re.IGNORECASE)
    return m.group(1).strip() if m else s


def _parse_time_to_seconds(time_str: str):
    if not isinstance(time_str, str):
        return None
    s = time_str.strip()
    if not s:
        return None

    parts = s.split(":")
    try:
        if len(parts) == 3:
            h, m, sec = parts
            return int(h) * 3600 + int(m) * 60 + float(sec)
        if len(parts) == 2:
            m, sec = parts
            return int(m) * 60 + float(sec)
    except ValueError:
        return None
    return None


def _seconds_to_hhmmss(seconds: float) -> str:
    total = max(0, int(round(seconds)))
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def _normalize_time_text(time_str: str) -> str | None:
    sec = _parse_time_to_seconds(time_str)
    if sec is None:
        return None
    return _seconds_to_hhmmss(sec)


def parse_model_json_strict(text):
    """ENACTING default: optional ``` fence strip, then json.loads only (no raw_decode / regex salvage)."""
    if isinstance(text, (dict, list)):
        return text
    if not isinstance(text, str):
        return None
    s = _unwrap_code_fence(text).strip()
    try:
        return json.loads(s)
    except Exception:
        return None


def safe_parse_json(text):
    # Already-structured payload from provider.
    if isinstance(text, (dict, list)):
        return text

    if not isinstance(text, str):
        return None

    s = _unwrap_code_fence(text).strip()

    # 1) Exact JSON.
    try:
        return json.loads(s)
    except Exception:
        pass

    # 2) First valid JSON value (allows trailing text).
    try:
        decoder = json.JSONDecoder()
        obj, _ = decoder.raw_decode(s)
        return obj
    except Exception:
        pass

    # 3) Fallback object/array span parse.
    for pat in (r"\{[\s\S]*\}", r"\[[\s\S]*\]"):
        m = re.search(pat, s)
        if not m:
            continue
        cand = m.group(0).strip()
        try:
            return json.loads(cand)
        except Exception:
            try:
                obj, _ = json.JSONDecoder().raw_decode(cand)
                return obj
            except Exception:
                pass

    return None


def normalize_output_data(data):
    """Normalize provider payload into (parsed_json_or_none, error_message_or_none, raw_text)."""
    if isinstance(data, dict):
        # Backend may wrap error in structured dict/list; coerce safely to text.
        err = data.get("error") or data.get("message")
        if err is not None:
            err_text = str(err).strip()
            if err_text:
                return None, err_text, json.dumps(data, ensure_ascii=False)
        return data, None, json.dumps(data, ensure_ascii=False)

    if isinstance(data, list):
        if len(data) == 0:
            return None, "Empty list response", "[]"
        return data, None, json.dumps(data, ensure_ascii=False)

    if isinstance(data, str):
        s = data.strip()
        if not s:
            return None, "Empty string response", ""
        if s.lower().startswith("error:"):
            return None, s, s
        parsed = safe_parse_json(s)
        return parsed, None if parsed is not None else None, s

    return None, f"Unsupported response type: {type(data).__name__}", str(data)



def _extract_screen_gaze_windows_from_csv(csv_text: str):
    df_local = pd.read_csv(pd.io.common.StringIO(csv_text)).fillna("")
    df_local["modality"] = df_local["modality"].astype(str)
    df_local["data"] = df_local["data"].astype(str)

    gaze = df_local[(df_local["modality"] == "gaze") & (df_local["data"] == "Screen")]
    windows = []
    for _, r in gaze.iterrows():
        s = _parse_time_to_seconds(str(r.get("start_time", "")))
        e = _parse_time_to_seconds(str(r.get("end_time", "")))
        if s is None or e is None:
            continue
        windows.append((s, e))
    return windows


def normalize_and_validate_segments(segments: list, csv_text: str):
    # Keep segments when times parse. No ordering rule (does not require time_in < time_out). Dedup + sort only.
    if not isinstance(segments, list):
        return []

    out = []
    seen = set()

    for seg in segments:
        if not isinstance(seg, dict):
            continue

        s = _parse_time_to_seconds(str(seg.get("time_in", "")))
        e = _parse_time_to_seconds(str(seg.get("time_out", "")))
        if s is None or e is None:
            continue

        row = {
            "justification": str(seg.get("justification", "")).strip(),
            "time_in": _seconds_to_hhmmss(s),
            "time_out": _seconds_to_hhmmss(e),
            "label": str(seg.get("label", "ENACTING") or "ENACTING"),
        }

        # Exact duplicate suppression only.
        dedup_key = (row["justification"], row["time_in"], row["time_out"], row["label"])
        if dedup_key in seen:
            continue
        seen.add(dedup_key)

        out.append(row)

    out.sort(key=lambda x: (
        _parse_time_to_seconds(x.get("time_in", "")),
        _parse_time_to_seconds(x.get("time_out", "")),
    ))
    return out


def compact_segment_text(segments: list, max_words: int = 12, max_chars: int = 90):
    """Hard-cap justification verbosity to keep model outputs compact and stable."""
    if not isinstance(segments, list):
        return []

    compacted = []
    for seg in segments:
        if not isinstance(seg, dict):
            continue

        j = str(seg.get("justification", "")).strip()
        if not j:
            j = "Verified event"

        # Keep only the first sentence-like chunk.
        j = re.split(r"(?<=[.!?])\s+", j)[0].strip()

        # Enforce short text budget.
        words = j.split()
        if len(words) > max_words:
            j = " ".join(words[:max_words])
        if len(j) > max_chars:
            j = j[:max_chars].rstrip()

        compacted.append({
            "justification": j,
            "time_in": seg.get("time_in", ""),
            "time_out": seg.get("time_out", ""),
            "label": seg.get("label", "ENACTING"),
        })

    return compacted


def sanitize_filename(text: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]", "_", text)


def run_pipeline(model_id: str, messages: list, base_url: str, headers: dict,
                 output_dir: str = ".", output_stem: str | None = None,
                 max_tokens: int = 4000, temperature: float = 0,
                 max_attempts: int = 3, strict_mode: bool = True,
                 segments_only: bool = True,
                 request_timeout: float = 600,
                 model_object: dict | None = None,
                 parse_json_lenient: bool = False):
    # ENACTING + INTERACTING: strict JSON only (parse_json_lenient=False) — no safe_parse_json salvage.

    def prepare_messages_for_model(_model_id_inner: str, msg_list: list):
        # Claude: first block must be user, not system (instructions → user; demo CSV already user).
        mid = (_model_id_inner or "").lower()
        if not ("claude" in mid or "anthropic" in mid):
            return msg_list
        out = []
        for i, item in enumerate(msg_list):
            if not isinstance(item, dict):
                out.append(item)
                continue
            role = str(item.get("role", "user"))
            if i == 0 and role == "system":
                out.append({**item, "role": "user"})
            else:
                out.append(item)
        return out

    def coerce_messages_to_text(msg_list: list):
        out = []
        for item in msg_list:
            role = str(item.get("role", "user"))
            content = item.get("content", "")
            if isinstance(content, str):
                text = content
            elif isinstance(content, (dict, list)):
                text = json.dumps(content, ensure_ascii=False)
            else:
                text = str(content)
            out.append({"role": role, "content": text})
        return out

    model_messages = coerce_messages_to_text(prepare_messages_for_model(model_id, messages))

    csv_text = ""
    for m in reversed(model_messages):
        if m.get("role") == "user" and "start_time" in str(m.get("content", "")):
            csv_text = str(m.get("content", ""))
            break

    tag = sanitize_filename(model_id)
    stem = (output_stem or f"MESSI_ENACT_{tag}").strip()
    json_path = os.path.join(output_dir, f"{stem}.json")

    # Use caller-provided token budget consistently across models.
    cap = max(1, int(max_tokens))
    n_sched = max(4, int(max_attempts))
    _out_tok_cap = 128000  # lower if the gateway rejects large max_tokens
    _mults = (1, 2, 3, 4)
    token_schedule = [
        min(cap * _mults[min(i, len(_mults) - 1)], _out_tok_cap)
        for i in range(n_sched)
    ]

    payload = {
        "data": {
            "temperature": temperature,
            "max_tokens": token_schedule[0],
            "dataSources": [],
            "messages": model_messages,
            "response_format": {"type": "json_object"},
            "options": {
                "skipRag": True,
                "ragOnly": False,
                "model": model_object if isinstance(model_object, dict) else {"id": model_id},
            },
        }
    }

    last_error = None
    used_model_object_fallback = isinstance(model_object, dict)
    for attempt in range(1, max_attempts + 1):
        token_idx = min(attempt - 1, len(token_schedule) - 1)
        payload["data"]["max_tokens"] = token_schedule[token_idx]

        response = requests.post(f"{base_url}/chat", headers=headers, json=payload, timeout=float(request_timeout))
        if response.status_code >= 400:
            body_text = response.text[:1000]
            print(f"[attempt {attempt}] HTTP {response.status_code} body:", body_text)
            if (not used_model_object_fallback) and response.status_code == 400:
                payload["data"]["options"]["model"] = {"id": model_id}
                used_model_object_fallback = True
                print(f"[attempt {attempt}] HTTP 400 -> switching model field to object fallback")
                continue
            last_error = RuntimeError(f"HTTP {response.status_code}: {body_text}")
            if response.status_code in (502, 503, 504, 529) and attempt < max_attempts:
                wait_s = min(120, 10 * attempt)
                print(f"[attempt {attempt}] transient HTTP {response.status_code}; sleeping {wait_s}s then retry")
                time.sleep(wait_s)
            continue

        result = response.json()

        print(f"[attempt {attempt}] max_tokens:", payload["data"]["max_tokens"])
        print(f"[attempt {attempt}] message_count:", len(payload["data"]["messages"]))
        print(f"[attempt {attempt}] result keys:", list(result.keys()) if isinstance(result, dict) else type(result).__name__)
        print(f"[attempt {attempt}] result.success:", result.get("success") if isinstance(result, dict) else None)
        print(f"[attempt {attempt}] repr(result.data):", repr(result.get("data")) if isinstance(result, dict) else None)

        if not isinstance(result, dict) or not result.get("success"):
            last_error = RuntimeError(f"Amplify returned failure: {result}")
            continue

        raw_data = result.get("data", "")
        if isinstance(raw_data, (dict, list)):
            raw_text = json.dumps(raw_data, ensure_ascii=False)
        else:
            raw_text = str(raw_data).strip()

        if not raw_text:
            last_error = RuntimeError(f"Empty response for {model_id}")
            continue

        if raw_text.lower().startswith("error:"):
            # Try object-style model fallback only for known gateway model-shape error.
            if (not used_model_object_fallback) and ("can only concatenate str" in raw_text):
                payload["data"]["options"]["model"] = {"id": model_id}
                used_model_object_fallback = True
                print(f"[attempt {attempt}] switching model field to object fallback for next retry")
                continue
            last_error = RuntimeError(raw_text)
            continue

        parsed_json = safe_parse_json(raw_text) if parse_json_lenient else parse_model_json_strict(raw_text)
        if parsed_json is None:
            last_error = RuntimeError(f"Strict mode: invalid JSON for {model_id}")
            continue

        if isinstance(parsed_json, dict) and "segments" in parsed_json:
            segments = parsed_json.get("segments", [])
        elif isinstance(parsed_json, list):
            segments = parsed_json
        else:
            segments = []

        if not isinstance(segments, list):
            segments = []

        segments = normalize_and_validate_segments(segments, csv_text)
        segments = compact_segment_text(segments)

        rule_based = False

        out_obj = {"segments": segments}
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(out_obj, f, indent=2, ensure_ascii=False)

        return {
            "json_path": json_path,
            "segment_count": len(segments),
            "rule_based": rule_based,
        }

    if last_error is not None:
        raise last_error
    raise RuntimeError(f"run_pipeline failed for {model_id} after {max_attempts} attempt(s)")


In [9]:

if "run_pipeline" in globals() and "_run_pipeline_original" not in globals():
    _run_pipeline_original = run_pipeline


def safe_parse_json(text):
    # Already-structured payload from provider.
    if isinstance(text, (dict, list)):
        return text
    if not isinstance(text, str):
        return None

    s = _unwrap_code_fence(text).strip()

    # 1) Exact JSON.
    try:
        return json.loads(s)
    except Exception:
        pass

    # 2) First valid JSON value (allows trailing prose).
    try:
        decoder = json.JSONDecoder()
        obj, _ = decoder.raw_decode(s)
        return obj
    except Exception:
        pass

    # 3) Fallback object/array span.
    for pat in (r"\{[\s\S]*\}", r"\[[\s\S]*\]"):
        m = re.search(pat, s)
        if not m:
            continue
        cand = m.group(0).strip()
        try:
            return json.loads(cand)
        except Exception:
            try:
                obj, _ = json.JSONDecoder().raw_decode(cand)
                return obj
            except Exception:
                pass

    return None


def run_pipeline(model_id: str, messages: list, base_url: str, headers: dict,
                 output_dir: str = ".", output_stem: str | None = None,
                 max_tokens: int = 4000, temperature: float = 0,
                 max_attempts: int = 3, strict_mode: bool = True,
                 segments_only: bool = True,
                 request_timeout: float = 600,
                 model_object: dict | None = None,
                 parse_json_lenient: bool = False):

    if "_run_pipeline_original" not in globals():
        raise RuntimeError("Hotfix loaded before base pipeline. Run the 'Unified Inference Pipeline' cell first, then rerun this hotfix cell.")

    def _call(mtok: int):
        return _run_pipeline_original(
            model_id=model_id,
            messages=messages,
            base_url=base_url,
            headers=headers,
            output_dir=output_dir,
            output_stem=output_stem,
            max_tokens=mtok,
            temperature=temperature,
            max_attempts=max_attempts,
            strict_mode=strict_mode,
            segments_only=segments_only,
            request_timeout=request_timeout,
            model_object=model_object,
            parse_json_lenient=parse_json_lenient,
        )

    mtok0 = int(max_tokens)
    try:
        return _call(mtok0)
    except RuntimeError as e:
        msg = str(e)
        if strict_mode and "Strict mode: invalid JSON" in msg:
            retry_tokens = min(max(mtok0 * 2, 16000), 128000)
            if retry_tokens > mtok0:
                return _call(retry_tokens)
        raise


print("Hotfix loaded: robust JSON parse; same max_tokens for all models; JSON-fail retry up to 128k")


Hotfix loaded: robust JSON parse; same max_tokens for all models; JSON-fail retry up to 128k


In [10]:
print("run_pipeline defined from:", run_pipeline.__code__.co_filename)
print("normalize_output_data exists:", "normalize_output_data" in globals())

run_pipeline defined from: /var/folders/kz/_0d2ww4n3wx819j1cz_zzvtw0000gn/T/ipykernel_30343/3429612417.py
normalize_output_data exists: True


## 6. Four models (ENACTING)

Optional batch of four models on global **`messages`**.


In [17]:
# Four models (ENACTING): one JSON per model -> MESSI_ENACT_<name>.json.
# Uses global `messages` (ENACTING few-shot + mastersheet CSV), not `messages_interacting`.
RUN_ENACTING_FOUR_MODELS = True  # set True to run ENACTING

if not RUN_ENACTING_FOUR_MODELS:
    print("Skip ENACTING: set RUN_ENACTING_FOUR_MODELS = True to run.")
else:
    available_model_ids, default_model_id, _available_payload, available_model_lookup = get_available_models(BASE_URL, headers)
    print("Default model:", default_model_id)
    print("Total available models:", len(available_model_ids))
    print("Available model ids:", available_model_ids)

    MAX_TOKENS = 64000
    MAX_ATTEMPTS = 3

    DEFAULT_BEHAVIOR_LABEL = "ENACTING"
    OUTPUT_STEM_PREFIX = "MESSI_ENACT"
    AMALIA_PROCESSOR_LABEL = "ENACTING MESSI"
    _P = OUTPUT_STEM_PREFIX
    ALL_MODEL_RUNS = [
        {"model_id": "us.anthropic.claude-sonnet-4-5-20250929-v1:0", "output_stem": f"{_P}_Claude_sonnet_4_5", "max_tokens": MAX_TOKENS},
        {"model_id": "o3", "output_stem": f"{_P}_o3", "max_tokens": MAX_TOKENS},
        {"model_id": "gpt-5.2", "output_stem": f"{_P}_ChatGPT_5_2", "max_tokens": MAX_TOKENS},
        {"model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "output_stem": f"{_P}_Claude_haiku_4_5", "max_tokens": MAX_TOKENS},
    ]

    MODEL_RUNS = ALL_MODEL_RUNS

    STRICT_MODE = True
    OUTPUT_DIR = ""
    import traceback

    reports = []
    offered_ids_lower = {m.lower().strip() for m in available_model_ids if isinstance(m, str)}

    for entry in MODEL_RUNS:
        mid = entry["model_id"]
        stem = entry["output_stem"]
        mtok = entry.get("max_tokens", 4000)

        if not isinstance(mid, str) or mid.lower().strip() not in offered_ids_lower:
            print(f"Skip (not offered by API): {mid} -> {stem}.json")
            continue

        try:
            print(f"\n=== {stem} ({mid}) ===")
            r = run_pipeline(
                model_id=mid,
                messages=messages,
                base_url=BASE_URL,
                headers=headers,
                output_dir=OUTPUT_DIR,
                output_stem=stem,
                max_tokens=mtok,
                temperature=0,
                max_attempts=MAX_ATTEMPTS,
                strict_mode=STRICT_MODE,
                segments_only=True,
                model_object=available_model_lookup.get(mid),
            )
            reports.append(r)
            print("Saved:", r["json_path"], "segments:", r.get("segment_count"), "rule_based:", r.get("rule_based"))
        except Exception as e:
            print(f"Failed: {mid} -> {e}")
            traceback.print_exc()

reports


Default model: us.anthropic.claude-haiku-4-5-20251001-v1:0
Total available models: 12
Available model ids: ['o3', 'gpt-4o', 'us.anthropic.claude-sonnet-4-20250514-v1:0', 'us.anthropic.claude-haiku-4-5-20251001-v1:0', 'gpt-4.1-mini', 'o4-mini', 'us.anthropic.claude-sonnet-4-6', 'us.anthropic.claude-3-5-haiku-20241022-v1:0', 'gpt-5', 'us.anthropic.claude-opus-4-6-v1', 'us.anthropic.claude-sonnet-4-5-20250929-v1:0', 'gpt-5.2']

=== MESSI_ENACT_Claude_sonnet_4_5 (us.anthropic.claude-sonnet-4-5-20250929-v1:0) ===
[attempt 1] max_tokens: 64000
[attempt 1] message_count: 4
[attempt 1] result keys: ['success', 'message', 'data']
[attempt 1] result.success: True
[attempt 1] repr(result.data): '```json\n{\n  "segments": [\n    {\n      "justification": "Condition 1: moving 0:00:06-0:00:08 with Screen gaze 0:00:07-0:04:38.",\n      "time_in": "00:00:07",\n      "time_out": "00:00:08",\n      "label": "ENACTING"\n    },\n    {\n      "justification": "Condition 1: moving 0:00:15-0:00:39 with Scree

[{'json_path': 'MESSI_ENACT_Claude_sonnet_4_5.json',
  'segment_count': 193,
  'rule_based': False},
 {'json_path': 'MESSI_ENACT_o3.json',
  'segment_count': 24,
  'rule_based': False},
 {'json_path': 'MESSI_ENACT_ChatGPT_5_2.json',
  'segment_count': 189,
  'rule_based': False},
 {'json_path': 'MESSI_ENACT_Claude_haiku_4_5.json',
  'segment_count': 192,
  'rule_based': False}]

## 7. Four models (INTERACTING)

Optional batch on **`messages_interacting`**.


In [26]:
# Optional: run four models on INTERACTING (Rose day 1) using `messages_interacting`.
# Requires earlier cells: BASE_URL/headers, get_available_models, Unified Inference Pipeline, and the INTERACTING block above.
# Uses MAX_TOKENS / MAX_ATTEMPTS when defined; otherwise 64000 / 3.
RUN_INTERACTING_FOUR_MODELS = True  # set True to execute this cell

if not RUN_INTERACTING_FOUR_MODELS:
    print("Skip: set RUN_INTERACTING_FOUR_MODELS = True to run.")
else:
    if "get_available_models" not in globals() or "run_pipeline" not in globals():
        raise RuntimeError("Run BASE_URL/headers, get_available_models, and Unified Pipeline cells first.")

    _mtok = int(globals().get("MAX_TOKENS", 64000))
    _matt = int(globals().get("MAX_ATTEMPTS", 3))

    DEFAULT_BEHAVIOR_LABEL = DEFAULT_BEHAVIOR_LABEL_INTER
    OUTPUT_STEM_PREFIX = OUTPUT_STEM_PREFIX_INTER
    AMALIA_PROCESSOR_LABEL = AMALIA_PROCESSOR_LABEL_INTER

    available_model_ids, default_model_id, _available_payload, available_model_lookup = get_available_models(
        BASE_URL, headers
    )
    offered_ids_lower = {m.lower().strip() for m in available_model_ids if isinstance(m, str)}

    _P = OUTPUT_STEM_PREFIX_INTER
    ALL_MODEL_RUNS_I = [
        {"model_id": "us.anthropic.claude-sonnet-4-5-20250929-v1:0", "output_stem": f"{_P}_Claude_sonnet_4_5", "max_tokens": _mtok},
        {"model_id": "o3", "output_stem": f"{_P}_o3", "max_tokens": _mtok},
        {"model_id": "gpt-5.2", "output_stem": f"{_P}_ChatGPT_5_2", "max_tokens": _mtok},
        {"model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "output_stem": f"{_P}_Claude_haiku_4_5", "max_tokens": _mtok},
    ]

    reports_inter = []
    for entry in ALL_MODEL_RUNS_I:
        mid = entry["model_id"]
        stem = entry["output_stem"]
        if not isinstance(mid, str) or mid.lower().strip() not in offered_ids_lower:
            print("Skip (not offered):", mid)
            continue
        print(f"\n=== INTER {stem} ({mid}) ===")
        r = run_pipeline(
            model_id=mid,
            messages=messages_interacting,
            base_url=BASE_URL,
            headers=headers,
            output_dir="",
            output_stem=stem,
            max_tokens=entry["max_tokens"],
            temperature=0,
            max_attempts=_matt,
            strict_mode=True,
            segments_only=True,
            model_object=available_model_lookup.get(mid),
            parse_json_lenient=False,
        )
        reports_inter.append(r)
        print("Saved:", r["json_path"], "segments:", r.get("segment_count"))

    DEFAULT_BEHAVIOR_LABEL = "ENACTING"
    OUTPUT_STEM_PREFIX = "MESSI_ENACT"
    AMALIA_PROCESSOR_LABEL = "ENACTING MESSI"
    print("Restored globals to ENACTING defaults for the rest of the notebook.")



=== INTER Rose_D1_INTER_Claude_sonnet_4_5 (us.anthropic.claude-sonnet-4-5-20250929-v1:0) ===
[attempt 1] max_tokens: 64000
[attempt 1] message_count: 4
[attempt 1] result keys: ['success', 'message', 'data']
[attempt 1] result.success: True
[attempt 1] repr(result.data): '```json\n{\n  "segments": [\n    {\n      "justification": "At time 00:02:29, Rose\'s [state] changes to [sugar]. According to Interacting Condition 1, transformation to [sugar] can only occur when paired up with another student, so Rose is INTERACTING.",\n      "time_in": "00:02:29",\n      "time_out": "00:02:29",\n      "label": "INTERACTING"\n    },\n    {\n      "justification": "At time 00:03:27, Rose says \'Okay, now you come with me so we can make like sugar.\' This [speech] is directed towards another student (SJ3747) and is relevant to the embodied activity (coordinating molecule combinations). According to Interacting Condition 2, Rose is INTERACTING.",\n      "time_in": "00:03:27",\n      "time_out": "00:0

## time inverter

In [ ]:
import os
COMPARE_BASE_DIR = ""
print("COMPARE_BASE_DIR:", COMPARE_BASE_DIR)

In [ ]:
import json
import os
import time
from datetime import timedelta


def time_to_seconds(time_str):
    if ":" in time_str:
        h, m, s = time_str.split(":")
        return int(h) * 3600 + int(m) * 60 + float(s)
    else:
        return float(time_str)


def seconds_to_tc(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    return f"{hours:02}:{minutes:02}:{secs:02}.0000"


def clean_segments(segments):
    processed = []

    for seg in segments:
        start = time_to_seconds(seg["time_in"])
        end = time_to_seconds(seg["time_out"])

        processed.append((start, end))

    processed.sort(key=lambda x: x[0])

    return processed


def convert_llm_to_amalia(input_file, output_file, metadata_id, label):

    with open(input_file, "r") as f:
        data = json.load(f)

    raw_segments = data["segments"]
    cleaned = clean_segments(raw_segments)

    # Build level1 list
    level1_segments = []

    for start, end in cleaned:
        level1_segments.append({
            "tclevel": 1,
            "tcin": seconds_to_tc(start),
            "tcout": seconds_to_tc(end),
            "label": label
        })

    amalia_format = {
        "type": "text",
        "id": metadata_id,
        "algorithm": "LLM",
        "processor": "ChatGPT",
        "processed": int(time.time() * 1000),
        "version": 1,
        "localisation": [
            {
                "sublocalisations": {
                    "localisation": level1_segments
                },
                "type": "text",
                "tcin": "00:00:00.0000",
                "tcout": "00:10:27.0000",
                "tclevel": 0
            }
        ]
    }

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w") as f:
        json.dump(amalia_format, f, indent=4)

    print("Generated:", output_file)
    print("Segments after merge:", len(cleaned))


if __name__ == "__main__":

    BASE_DIR = "/Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT"

    jobs = [
        (
            "MESSI_ENACT_ChatGPT_5_2.json",
            "samples-data/data-final/d2g2-segments-messi-gpt52-enact.json",
            "d2g2-segments-messi-gpt52-enact",
            "ENACTING MESSI",
        ),
        (
            "MESSI_ENACT_o3.json",
            "samples-data/data-final/d2g2-segments-messi-o3-enact.json",
            "d2g2-segments-messi-o3-enact",
            "ENACTING MESSI",
        ),
        (
            "MESSI_ENACT_Claude_haiku_4_5.json",
            "samples-data/data-final/d2g2-segments-messi-claude-haiku-4-5-enact.json",
            "d2g2-segments-messi-claude-haiku-4-5-enact",
            "ENACTING MESSI",
        ),
        (
            "MESSI_ENACT_Claude_sonnet_4_5.json",
            "samples-data/data-final/d2g2-segments-messi-claude-sonnet-4-5-enact.json",
            "d2g2-segments-messi-claude-sonnet-4-5-enact",
            "ENACTING MESSI",
        ),
    ]

    for input_file, output_file, metadata_id, label in jobs:
        input_path = os.path.join(BASE_DIR, input_file)
        output_path = os.path.join(BASE_DIR, output_file)

        if not os.path.exists(input_path):
            print(f"Skip missing file: {input_path}")
            continue

        convert_llm_to_amalia(input_path, output_path, metadata_id, label)


## Segment Boundary Comparison (Merged)
Merged from `segment_boundary_comparison.ipynb`.

In [ ]:
import os
COMPARE_BASE_DIR = ""
os.path.abspath(COMPARE_BASE_DIR)

In [ ]:
with open(os.path.join(COMPARE_BASE_DIR, "samples-data/data-final/d2g2-segments-messi-gpt52-enact.json")) as f:
    data = json.load(f)

print(type(data))
print(data)


In [ ]:
# Method1: Multi-model all-pairs comparison
import json
import itertools
from collections import Counter
import pandas as pd


def classify(a_start, a_end, b_start, b_end):
    if a_start == b_start and a_end == b_end:
        return "EXACT_MATCH"
    if a_start == b_start:
        return "SAME_START"
    if a_end == b_end:
        return "SAME_END"
    if b_start <= a_start and b_end >= a_end:
        return "B_CONTAINS_A"
    if a_start <= b_start and a_end >= b_end:
        return "A_CONTAINS_B"
    if a_start < b_end and a_end > b_start:
        return "PARTIAL_OVERLAP"
    return "NO_OVERLAP"


def tc_to_seconds(tc):
    h, m, s = tc.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


MODEL_FILES = {
    "gpt52": "samples-data/data-final/d2g2-segments-messi-gpt52-enact.json",
    "o3": "samples-data/data-final/d2g2-segments-messi-o3-enact.json",
    "claude_haiku": "samples-data/data-final/d2g2-segments-messi-claude-haiku-4-5-enact.json",
    "claude_sonnet": "samples-data/data-final/d2g2-segments-messi-claude-sonnet-4-5-enact.json",
}

model_segments = {}
for model_name, rel_path in MODEL_FILES.items():
    abs_path = os.path.join(COMPARE_BASE_DIR, rel_path)
    if not os.path.exists(abs_path):
        print(f"Skip missing file for {model_name}: {abs_path}")
        continue
    with open(abs_path) as f:
        data = json.load(f)
    segs = data["localisation"][0]["sublocalisations"]["localisation"]
    model_segments[model_name] = segs

print("Loaded models and segment counts:")
for k, v in model_segments.items():
    print(f"- {k}: {len(v)}")

pairwise_relation_stats = []
pairwise_results = {}

for model_a, model_b in itertools.combinations(model_segments.keys(), 2):
    results = []
    segs_a = model_segments[model_a]
    segs_b = model_segments[model_b]

    for seg_a in segs_a:
        a_start = tc_to_seconds(seg_a["tcin"])
        a_end = tc_to_seconds(seg_a["tcout"])
        for seg_b in segs_b:
            b_start = tc_to_seconds(seg_b["tcin"])
            b_end = tc_to_seconds(seg_b["tcout"])
            rel = classify(a_start, a_end, b_start, b_end)
            results.append({
                "model_a": model_a,
                "model_b": model_b,
                "a": [a_start, a_end],
                "b": [b_start, b_end],
                "relation": rel,
            })

    pair_key = f"{model_a}__vs__{model_b}"
    pairwise_results[pair_key] = results

    c = Counter(r["relation"] for r in results)
    row = {"pair": pair_key, "total": len(results)}
    row.update(c)
    pairwise_relation_stats.append(row)

pairwise_stats_df = pd.DataFrame(pairwise_relation_stats).fillna(0)
pairwise_stats_df = pairwise_stats_df.sort_values("pair").reset_index(drop=True)
pairwise_stats_df


In [ ]:
# Optional: inspect one pair in detail
PAIR_TO_INSPECT = "gpt52__vs__claude_sonnet"

if PAIR_TO_INSPECT in pairwise_results:
    sample = pairwise_results[PAIR_TO_INSPECT][:20]
    print(f"Pair: {PAIR_TO_INSPECT} | sample rows: {len(sample)}")
    for r in sample:
        print(r)
else:
    print(f"Pair not found: {PAIR_TO_INSPECT}")


In [ ]:
# Method2: Best-match mapping for any chosen pair
MODEL_A = "gpt52"
MODEL_B = "claude_sonnet"


def overlap(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))


best_matches = []

if MODEL_A in model_segments and MODEL_B in model_segments:
    segs_a = model_segments[MODEL_A]
    segs_b = model_segments[MODEL_B]

    for i, seg_a in enumerate(segs_a):
        a_start = tc_to_seconds(seg_a["tcin"])
        a_end = tc_to_seconds(seg_a["tcout"])

        best_overlap = -1
        best_seg = None
        best_j = None

        for j, seg_b in enumerate(segs_b):
            b_start = tc_to_seconds(seg_b["tcin"])
            b_end = tc_to_seconds(seg_b["tcout"])
            ov = overlap(a_start, a_end, b_start, b_end)
            if ov > best_overlap:
                best_overlap = ov
                best_seg = (b_start, b_end)
                best_j = j + 1

        relation = classify(a_start, a_end, best_seg[0], best_seg[1]) if best_seg else "NO_MATCH"
        best_matches.append({
            "A_model": MODEL_A,
            "A_id": i + 1,
            "A_segment": [a_start, a_end],
            "B_model": MODEL_B,
            "B_id": best_j,
            "B_segment": best_seg,
            "best_overlap": best_overlap,
            "relation": relation,
        })

len(best_matches)


In [ ]:
print("Total best matches:", len(best_matches))
for m in best_matches[:30]:
    print(m)

In [ ]:
# Method3: One-to-many overlap mapping for any chosen pair
one_to_many = []

if MODEL_A in model_segments and MODEL_B in model_segments:
    segs_a = model_segments[MODEL_A]
    segs_b = model_segments[MODEL_B]

    for i, seg_a in enumerate(segs_a):
        a_start = tc_to_seconds(seg_a["tcin"])
        a_end = tc_to_seconds(seg_a["tcout"])

        related_segments = []
        for j, seg_b in enumerate(segs_b):
            b_start = tc_to_seconds(seg_b["tcin"])
            b_end = tc_to_seconds(seg_b["tcout"])
            relation = classify(a_start, a_end, b_start, b_end)
            if relation != "NO_OVERLAP":
                related_segments.append({
                    "B_id": j + 1,
                    "segment": [b_start, b_end],
                    "relation": relation,
                })

        one_to_many.append({
            "A_id": i + 1,
            "A_segment": [a_start, a_end],
            "related_B": related_segments,
        })

len(one_to_many)

In [ ]:
for r in one_to_many[:3]:
    print(f"{MODEL_A}_", r["A_id"], r["A_segment"])
    for seg in r["related_B"]:
        print(f"   → {MODEL_B}_", seg["B_id"], seg["segment"], seg["relation"])

In [ ]:
# Prep for Method4 visualization (multi-model)
import matplotlib.pyplot as plt

boundary_data = {}
for model_name, segs in model_segments.items():
    starts = [tc_to_seconds(seg["tcin"]) for seg in segs]
    ends = [tc_to_seconds(seg["tcout"]) for seg in segs]
    boundary_data[model_name] = {"starts": starts, "ends": ends}

boundary_data.keys()

In [ ]:
## Method4: Segment boundary alignment visualization (multi-model)
plt.figure(figsize=(14, 6))

model_order = list(boundary_data.keys())
if not model_order:
    raise ValueError("No model data loaded for visualization.")

row_gap = 1.2
for i, model_name in enumerate(model_order):
    y_start = (len(model_order) - i) * row_gap
    y_end = y_start - 0.45

    starts = boundary_data[model_name]["starts"]
    ends = boundary_data[model_name]["ends"]

    plt.scatter(starts, [y_start] * len(starts), s=30, label=f"{model_name} start")
    plt.scatter(ends, [y_end] * len(ends), s=30, marker="x", label=f"{model_name} end")

plt.xlabel("Time (seconds)")
plt.title("Segment boundary comparison across models")

from matplotlib.ticker import MultipleLocator
ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(50))
ax.xaxis.set_minor_locator(MultipleLocator(10))
ax.grid(which="major", linestyle="-", linewidth=0.8, alpha=0.6)
ax.grid(which="minor", linestyle=":", linewidth=0.5, alpha=0.6)

plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.tight_layout()
plt.show()